In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
class InputEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model=512):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

# Test
embed = InputEmbeddings(vocab_size=1000, d_model=512)
tokens = torch.tensor([[1, 5, 3, 7]])
out = embed(tokens)
print("Token IDs shape  :", tokens.shape)   # [1, 4]
print("Embedding shape  :", out.shape)      # [1, 4, 512]

Token IDs shape  : torch.Size([1, 4])
Embedding shape  : torch.Size([1, 4, 512])


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model=512, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)   # even → sin
        pe[:, 1::2] = torch.cos(position * div_term)   # odd  → cos
        pe = pe.unsqueeze(0)                            # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# Test
pos_enc = PositionalEncoding(d_model=512)
x = torch.zeros(1, 4, 512)
out = pos_enc(x)
print("After Positional Encoding:", out.shape)   # [1, 4, 512]

After Positional Encoding: torch.Size([1, 4, 512])


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)

    # Step 1: scores = Q × K^T
    scores = torch.matmul(Q, K.transpose(-2, -1))   # [batch, heads, seq, seq]

    # Step 2: scale
    scores = scores / math.sqrt(d_k)

    # Step 3: apply mask
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # Step 4: softmax
    attn_weights = F.softmax(scores, dim=-1)

    # Step 5: weighted sum of V
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

# Test
Q = torch.randn(1, 8, 4, 64)
K = torch.randn(1, 8, 4, 64)
V = torch.randn(1, 8, 4, 64)
out, weights = scaled_dot_product_attention(Q, K, V)
print("Attention output :", out.shape)      # [1, 8, 4, 64]
print("Attention weights:", weights.shape)  # [1, 8, 4, 4]
print("Row sum (must=1) :", weights[0,0,0].sum().item())

Attention output : torch.Size([1, 8, 4, 64])
Attention weights: torch.Size([1, 8, 4, 4])
Row sum (must=1) : 1.0


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads   # 64

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def split_heads(self, x, batch):
        x = x.view(batch, -1, self.num_heads, self.d_k)
        return x.transpose(1, 2)   # [batch, heads, seq, d_k]

    def forward(self, Q, K, V, mask=None):
        batch = Q.size(0)
        Q = self.split_heads(self.W_Q(Q), batch)
        K = self.split_heads(self.W_K(K), batch)
        V = self.split_heads(self.W_V(V), batch)

        attn_out, _ = scaled_dot_product_attention(Q, K, V, mask)

        attn_out = attn_out.transpose(1, 2).contiguous()
        attn_out = attn_out.view(batch, -1, self.d_model)
        return self.W_O(attn_out)

# Test
mha = MultiHeadAttention(d_model=512, num_heads=8)
x = torch.randn(1, 4, 512)
out = mha(x, x, x)
print("Multi-Head Attention output:", out.shape)   # [1, 4, 512]

Multi-Head Attention output: torch.Size([1, 4, 512])


In [ ]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)    # 512 → 2048
        self.linear2 = nn.Linear(d_ff, d_model)    # 2048 → 512
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

# Test
ffn = FeedForwardNetwork()
x = torch.randn(1, 4, 512)
out = ffn(x)
print("FFN input :", x.shape)    # [1, 4, 512]
print("FFN output:", out.shape)  # [1, 4, 512]

FFN input : torch.Size([1, 4, 512])
FFN output: torch.Size([1, 4, 512])


In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn       = FeedForwardNetwork(d_model, d_ff, dropout)
        self.norm1     = nn.LayerNorm(d_model)
        self.norm2     = nn.LayerNorm(d_model)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Sub-layer 1: Self-Attention + Residual + Norm
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        # Sub-layer 2: FFN + Residual + Norm
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

# Test
enc_layer = EncoderLayer()
x = torch.randn(1, 4, 512)
out = enc_layer(x)
print("Encoder Layer output:", out.shape)   # [1, 4, 512]

Encoder Layer output: torch.Size([1, 4, 512])


In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.masked_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn  = MultiHeadAttention(d_model, num_heads)
        self.ffn         = FeedForwardNetwork(d_model, d_ff, dropout)
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.norm3   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        # Sub-layer 1: Masked Self-Attention
        x = self.norm1(x + self.dropout(self.masked_attn(x, x, x, tgt_mask)))
        # Sub-layer 2: Cross-Attention (Q=decoder, K/V=encoder)
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, src_mask)))
        # Sub-layer 3: FFN
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x

# Test
dec_layer = DecoderLayer()
tgt = torch.randn(1, 3, 512)
enc_out = torch.randn(1, 4, 512)
out = dec_layer(tgt, enc_out)
print("Decoder Layer output:", out.shape)   # [1, 3, 512]

Decoder Layer output: torch.Size([1, 3, 512])


In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=8,
                 d_ff=2048, num_layers=6, dropout=0.1):
        super().__init__()
        self.embedding = InputEmbeddings(vocab_size, d_model)
        self.pos_enc   = PositionalEncoding(d_model, dropout=dropout)
        self.layers    = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = self.pos_enc(self.embedding(x))
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

# Test
encoder = Encoder(vocab_size=1000)
src = torch.randint(0, 1000, (1, 4))
enc_out = encoder(src)
print("Encoder stack output:", enc_out.shape)   # [1, 4, 512]

Encoder stack output: torch.Size([1, 4, 512])


In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=8,
                 d_ff=2048, num_layers=6, dropout=0.1):
        super().__init__()
        self.embedding = InputEmbeddings(vocab_size, d_model)
        self.pos_enc   = PositionalEncoding(d_model, dropout=dropout)
        self.layers    = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        x = self.pos_enc(self.embedding(x))
        for layer in self.layers:
            x = layer(x, enc_out, src_mask, tgt_mask)
        return self.norm(x)

# Test
decoder = Decoder(vocab_size=1000)
tgt = torch.randint(0, 1000, (1, 3))
dec_out = decoder(tgt, enc_out)
print("Decoder stack output:", dec_out.shape)   # [1, 3, 512]

Decoder stack output: torch.Size([1, 3, 512])


In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab,
                 d_model=512, num_heads=8,
                 d_ff=2048, num_layers=6, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(src_vocab, d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(tgt_vocab, d_model, num_heads, d_ff, num_layers, dropout)
        self.fc_out  = nn.Linear(d_model, tgt_vocab)

    def make_causal_mask(self, size):
        # Lower triangular mask — decoder can't see future tokens
        mask = torch.tril(torch.ones(size, size)).unsqueeze(0).unsqueeze(0)
        return mask   # [1, 1, size, size]

    def forward(self, src, tgt, src_mask=None):
        tgt_mask   = self.make_causal_mask(tgt.size(1)).to(tgt.device)
        enc_out    = self.encoder(src, src_mask)
        dec_out    = self.decoder(tgt, enc_out, src_mask, tgt_mask)
        return self.fc_out(dec_out)   # [batch, tgt_seq_len, tgt_vocab]

# Test
model = Transformer(src_vocab=1000, tgt_vocab=1000)
src   = torch.randint(0, 1000, (1, 4))
tgt   = torch.randint(0, 1000, (1, 3))
out   = model(src, tgt)
print("Final output shape:", out.shape)   # [1, 3, 1000]
print("→ logits over 1000 vocab tokens for each of 3 target positions")

Final output shape: torch.Size([1, 3, 1000])
→ logits over 1000 vocab tokens for each of 3 target positions
